# First look (exploratory only)

Nothing in this notebook is part of the pipeline.

In [1]:
import pandas as pd
print("exploratory scratchpad")

exploratory scratchpad


In [2]:
# Load the input file
ids_df = pd.read_csv("../data/input/id_list.csv")



In [3]:
# Inspect the structure
print("Shape:", ids_df.shape)
print("\nColumn names:")
print(ids_df.columns.tolist())



Shape: (126, 1)

Column names:
['tmdb_id']


In [4]:
print("\nFirst five rows:")
print(ids_df.head())




First five rows:
  tmdb_id
0     155
1     165
2     242
3   49026
4    8077


In [5]:
print("\nData types:")
print(ids_df.dtypes)

print("\nMissing values:")
print(ids_df.isna().sum())

print("\nDuplicate rows:")
print(ids_df.duplicated().sum())


Data types:
tmdb_id    object
dtype: object

Missing values:
tmdb_id    1
dtype: int64

Duplicate rows:
6


In [6]:
# Combining the above into a single block for better readability
print("Shape:", ids_df.shape)
print("\nColumn names:")
print(ids_df.columns.tolist())

print("\nFirst five rows:")
print(ids_df.head())

print("\nData types:")
print(ids_df.dtypes)

print("\nMissing values:")
print(ids_df.isna().sum())

print("\nDuplicate rows:")
print(ids_df.duplicated().sum())

Shape: (126, 1)

Column names:
['tmdb_id']

First five rows:
  tmdb_id
0     155
1     165
2     242
3   49026
4    8077

Data types:
tmdb_id    object
dtype: object

Missing values:
tmdb_id    1
dtype: int64

Duplicate rows:
6


In [7]:
# Inspect rows with missing IDs
# we are filtering the DataFrame to show only rows where the "tmdb_id" column is NaN (missing)
print("Missing ID rows:")
print(ids_df[ids_df["tmdb_id"].isna()])



Missing ID rows:
    tmdb_id
108     NaN


In [8]:
# Inspect duplicated IDs
print("\nDuplicated ID rows:")
print(
    ids_df[
        ids_df["tmdb_id"].duplicated(keep=False)
    ].sort_values("tmdb_id")
)




Duplicated ID rows:
   tmdb_id
0      155
88     155
53    1891
87    1891
25     278
58     278
77     550
91     550
38     603
41     603
6      680
99     680


In [9]:

# Count unique non-missing IDs
print("\nUnique non-missing IDs:")
print(ids_df["tmdb_id"].nunique())



Unique non-missing IDs:
119


In [10]:
# Record missing rows
missing_input = ids_df[ids_df["tmdb_id"].isna()].copy()

# Record duplicate rows
duplicate_input = ids_df[
    ids_df["tmdb_id"].duplicated(keep=False)
].copy()



In [11]:
# Find IDs that are not purely numeric
numeric_mask = ids_df["tmdb_id"].astype("string").str.fullmatch(r"\d+")

print("Non-numeric IDs:")
print(ids_df.loc[~numeric_mask, "tmdb_id"])

Non-numeric IDs:
28           -1
92    tt0468569
Name: tmdb_id, dtype: object


In [12]:
invalid_ids = ids_df.loc[~numeric_mask].copy()

print(invalid_ids)

      tmdb_id
28         -1
92  tt0468569


In [13]:
# Identify numeric IDs
numeric_mask = (
    ids_df["tmdb_id"]
    .astype("string")
    .str.fullmatch(r"\d+")
)

# Keep invalid records for investigation
invalid_input = ids_df.loc[~numeric_mask].copy()

# Keep valid numeric IDs
clean_ids = ids_df.loc[numeric_mask].copy()

# Convert to integer
clean_ids["tmdb_id"] = clean_ids["tmdb_id"].astype("int64")

# Exclude non-positive IDs
clean_ids = clean_ids.loc[clean_ids["tmdb_id"] > 0]

# Identify and preserve duplicate records
"""duplicate_input = clean_ids[
    clean_ids["tmdb_id"].duplicated(keep=False)
].copy()"""

# Remove duplicate IDs
clean_ids = clean_ids.drop_duplicates(subset=["tmdb_id"])

print("Unique valid IDs:", len(clean_ids))
print("Invalid input records:", len(invalid_input))
print("Duplicate records:", len(duplicate_input))

Unique valid IDs: 117
Invalid input records: 2
Duplicate records: 12


In [19]:
import requests
import sys, os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
from src.fetch import fetch_movie

session = requests.Session()

movie = fetch_movie(session, 603)

print(movie.get("title"))
print(movie.get("release_date"))
print(movie.get("revenue"))

The Matrix
1999-03-31
463517383


In [21]:
import importlib
import src.fetch as fetch_module

importlib.reload(fetch_module)

fetch_movie = fetch_module.fetch_movie

In [22]:
# Existing movie
movie = fetch_movie(session, 603)
print("Existing movie:", movie.get("title") if movie else None)

# Deliberately nonexistent movie ID
missing_movie = fetch_movie(session, 999999999)
print("Missing movie:", missing_movie)

Existing movie: The Matrix
Missing movie: None


In [23]:
import requests

records = []
missing_ids = []
failed_ids = []

test_ids = clean_ids["tmdb_id"].head(5)

for movie_id in test_ids:
    try:
        movie = fetch_movie(session, int(movie_id))

        if movie is None:
            missing_ids.append(movie_id)
        else:
            records.append(movie)

    except requests.RequestException as error:
        failed_ids.append({
            "tmdb_id": movie_id,
            "error": str(error)
        })



In [24]:
print("Requested:", len(test_ids))
print("Successful:", len(records))
print("Missing:", len(missing_ids))
print("Failed:", len(failed_ids))

Requested: 5
Successful: 5
Missing: 0
Failed: 0


In [25]:
import json
import os
from pathlib import Path

os.makedirs("../data/raw/details", exist_ok=True)

records = []
missing_ids = []
failed_ids = []

for movie_id in clean_ids["tmdb_id"]:
    try:
        movie = fetch_movie(session, int(movie_id))

        if movie is None:
            missing_ids.append(movie_id)
        else:
            records.append(movie)
            out_path = Path("../data/raw/details") / f"{movie_id}.json"
            with out_path.open("w", encoding="utf-8") as f:
                json.dump(movie, f, indent=2)

    except requests.RequestException as error:
        failed_ids.append({"tmdb_id": movie_id, "error": str(error)})

print("Requested:", len(clean_ids))
print("Successful:", len(records))
print("Missing:", len(missing_ids))
print("Failed:", len(failed_ids))
print("Saved to: data/raw/details/")

Requested: 117
Successful: 112
Missing: 5
Failed: 0
Saved to: data/raw/details/


In [26]:
missing_ids

[8471293, 9412887, 8471294, 9412888, 7733901]

In [29]:
from pathlib import Path

raw_dir = Path("../data/raw/details")

files = list(raw_dir.glob("*.json"))

print("Number of JSON files:", len(files))
print("\nFirst 5 files:")

for file in files[:5]:
    print(file.name)

Number of JSON files: 112

First 5 files:
10138.json
10192.json
10193.json
105.json
10681.json


In [32]:
import json

sample_file = files[0]

with open(sample_file, "r", encoding="utf-8") as f:
    movie = json.load(f)

print(json.dumps(movie, indent=4))

{
    "adult": false,
    "backdrop_path": "/7lmBufEG7P7Y1HClYK3gCxYrkgS.jpg",
    "belongs_to_collection": {
        "id": 131292,
        "name": "Iron Man Collection",
        "poster_path": "/fbeJ7f0aD4A112Bc1tnpzyn82xO.jpg",
        "backdrop_path": "/rI8zOWkRQJdlAyQ6WJOSlYK6JxZ.jpg"
    },
    "budget": 200000000,
    "genres": [
        {
            "id": 12,
            "name": "Adventure"
        },
        {
            "id": 28,
            "name": "Action"
        },
        {
            "id": 878,
            "name": "Science Fiction"
        }
    ],
    "homepage": "https://www.marvel.com/movies/iron-man-2",
    "id": 10138,
    "imdb_id": "tt1228705",
    "origin_country": [
        "US"
    ],
    "original_language": "en",
    "original_title": "Iron Man 2",
    "overview": "With the world now aware of his dual life as the armored superhero Iron Man, billionaire inventor Tony Stark faces pressure from the government, the press and the public to share his technology 

In [33]:
clean_ids

,tmdb_id
0,155
1,165
2,242
3,49026
4,8077
...,...
121,166426
122,157336
123,620
124,296


In [34]:

import json
import requests
from pathlib import Path
import importlib
import src.fetch as fetch_module

importlib.reload(fetch_module)

fetch_credits = fetch_module.fetch_credits

details_dir = Path("../data/raw/details")
credits_dir = Path("../data/raw/credits")
credits_dir.mkdir(parents=True, exist_ok=True)

# Reuse IDs from successfully saved movie details
movie_ids = [int(p.stem) for p in details_dir.glob("*.json")]

credits_success = []
credits_missing = []
credits_failed = []

for movie_id in movie_ids:
    try:
        credits = fetch_credits(session, movie_id)

        if credits is None:
            credits_missing.append(movie_id)
            continue

        with open(credits_dir / f"{movie_id}.json", "w") as f:
            json.dump(credits, f, indent=4)

        credits_success.append(movie_id)

    except requests.RequestException as error:
        credits_failed.append({
            "tmdb_id": movie_id,
            "error": str(error)
        })

print("Requested:", len(movie_ids))
print("Successful:", len(credits_success))
print("Missing:", len(credits_missing))
print("Failed:", len(credits_failed))
print("Saved to:", credits_dir)


Requested: 112
Successful: 112
Missing: 0
Failed: 0
Saved to: ..\data\raw\credits


In [35]:
import json
from pathlib import Path

details_file = next(Path("../data/raw/details").glob("*.json"))
credits_file = next(Path("../data/raw/credits").glob("*.json"))

with open(details_file) as f:
    details = json.load(f)

with open(credits_file) as f:
    credits = json.load(f)

print("DETAILS KEYS:")
print(details.keys())

print("\nCREDITS KEYS:")
print(credits.keys())

print("\nCAST SAMPLE:")
print(credits["cast"][:2])

print("\nCREW SAMPLE:")
print(credits["crew"][:2])

DETAILS KEYS:
dict_keys(['adult', 'backdrop_path', 'belongs_to_collection', 'budget', 'genres', 'homepage', 'id', 'imdb_id', 'origin_country', 'original_language', 'original_title', 'overview', 'popularity', 'poster_path', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'softcore', 'spoken_languages', 'status', 'tagline', 'title', 'video', 'vote_average', 'vote_count'])

CREDITS KEYS:
dict_keys(['id', 'cast', 'crew'])

CAST SAMPLE:
[{'adult': False, 'gender': 2, 'id': 3223, 'known_for_department': 'Acting', 'name': 'Robert Downey Jr.', 'original_name': 'Robert Downey Jr.', 'popularity': 10.2234, 'profile_path': '/5qHNjhtjMD4YWH3UP0rm4tKwxCL.jpg', 'cast_id': 30, 'character': 'Tony Stark', 'credit_id': '52fe43329251416c75007341', 'order': 0}, {'adult': False, 'gender': 1, 'id': 12052, 'known_for_department': 'Acting', 'name': 'Gwyneth Paltrow', 'original_name': 'Gwyneth Paltrow', 'popularity': 2.8033, 'profile_path': '/r4wbWWEEjmtRFZ0GU10XbLTgp47.jpg

In [36]:
import json
from pathlib import Path

details_file = sorted(Path("../data/raw/details").glob("*.json"))[0]
credits_file = sorted(Path("../data/raw/credits").glob("*.json"))[0]

with open(details_file) as f:
    d = json.load(f)
with open(credits_file) as f:
    c = json.load(f)

print("DETAILS keys:")
for k, v in d.items():
    print(f"  {k}: {type(v).__name__} = {str(v)[:60]}")

print("\nCREDITS top-level keys:", list(c.keys()))
print("cast entries:", len(c["cast"]), "| crew entries:", len(c["crew"]))
print("\nDirector entries (job == 'Director'):")
for person in c["crew"]:
    if person.get("job") == "Director":
        print(" ", person.get("name"), "|", person.get("id"))

DETAILS keys:
  adult: bool = False
  backdrop_path: str = /7lmBufEG7P7Y1HClYK3gCxYrkgS.jpg
  belongs_to_collection: dict = {'id': 131292, 'name': 'Iron Man Collection', 'poster_path':
  budget: int = 200000000
  genres: list = [{'id': 12, 'name': 'Adventure'}, {'id': 28, 'name': 'Action
  homepage: str = https://www.marvel.com/movies/iron-man-2
  id: int = 10138
  imdb_id: str = tt1228705
  origin_country: list = ['US']
  original_language: str = en
  original_title: str = Iron Man 2
  overview: str = With the world now aware of his dual life as the armored sup
  popularity: float = 21.1095
  poster_path: str = /6WBeq4fCfn7AN0o21W9qNcRF2l9.jpg
  production_companies: list = [{'id': 420, 'logo_path': '/mIkZDuulwMPzESbzF9lg3rD8CcO.png'
  production_countries: list = [{'iso_3166_1': 'US', 'name': 'United States of America'}]
  release_date: str = 2010-04-28
  revenue: int = 623933331
  runtime: int = 124
  softcore: bool = False
  spoken_languages: list = [{'english_name': 'English', 'is

In [37]:
import json
from pathlib import Path
import pandas as pd

# 1. The 5 ids that returned 404 (from your fetch loop's missing_ids)
print("Missing (404) IDs:", missing_ids)
for mid in missing_ids:
    print(f"  {mid} -> https://www.themoviedb.org/movie/{mid}")

# 2. Redirects: any film whose returned id != requested id
details_dir = Path("../data/raw/details")
redirects = []
for f in details_dir.glob("*.json"):
    with open(f) as fh:
        m = json.load(fh)
    if m.get("id") != int(f.stem):
        redirects.append((int(f.stem), m.get("id"), m.get("title")))
print("\nRedirects (requested id -> returned id):", redirects)

# 3. Cross-source duplicates: same imdb_id, or same (title, release year)
movies = []
for f in details_dir.glob("*.json"):
    with open(f) as fh:
        m = json.load(fh)
    movies.append({"tmdb_id": m.get("id"), "imdb_id": m.get("imdb_id"),
                   "title": m.get("title"), "year": (m.get("release_date") or "")[:4]})
mdf = pd.DataFrame(movies)

by_imdb = mdf[mdf["imdb_id"].notna() & mdf["imdb_id"].duplicated(keep=False)]
print("\nDuplicate imdb_id groups:", by_imdb.sort_values("imdb_id").to_dict("records"))

by_title_year = mdf[mdf.duplicated(subset=["title", "year"], keep=False)]
print("Duplicate (title, year) groups:", by_title_year.sort_values(["title", "year"]).to_dict("records"))

Missing (404) IDs: [8471293, 9412887, 8471294, 9412888, 7733901]
  8471293 -> https://www.themoviedb.org/movie/8471293
  9412887 -> https://www.themoviedb.org/movie/9412887
  8471294 -> https://www.themoviedb.org/movie/8471294
  9412888 -> https://www.themoviedb.org/movie/9412888
  7733901 -> https://www.themoviedb.org/movie/7733901

Redirects (requested id -> returned id): []

Duplicate imdb_id groups: []
Duplicate (title, year) groups: []
